# Scenario 1: <sup>1</sup>H-NMR kinetic analysis of Pd(0)-catalyzed cycloisomerization

## Data Processing and Analysis Notebook

---

### Notebook Setup

The essential packages to work with NMRPy and the EnzymeML extension are:

- `matplotlib`
- `nmrpy`
- `pyenzyme`

The following packages in this Notebook are used for convenience:

- `pathlib`
- `rich`

Depending on the use case, some additional packages maybe useful:

- `numpy`
- `pandas`
- `scipy`
- `pickle`
- `...`

Interactive plotting requires either the `TkAgg` or `ipympl` backend. While the `ipympl` backend allows for convenience interactive inline plotting in the Jupyter notebook, it does not run as smoothly as the `TkAgg` backend that pops up the graphs in separate windows. If interactivity is not required, `matplotlib.use("ipympl")` can be commented out and uncommenting `%matplotlib inline` will plot static graphs inline instead.

In [ ]:
from pathlib import Path
from rich import print
from rich.pretty import Pretty

import matplotlib
from matplotlib import pyplot as plt
import numpy as np

import nmrpy
import pyenzyme as pe

plt.ioff()
matplotlib.use("ipympl")
# %matplotlib inline

---

### Processing of the NMR Data

#### NMR data loading

An NMRPy workflow starts with the loading of the NMR data. The NMRPy package supports a variety of file formats, including Bruker, Varian, and more. When an EnzymeML file is used, like in this scenario, it can be loaded into NMRPy using the `parse_enzymeml_document` function, which extracts the species and measurement metadata and links it with the internal NMRPy data model.

In [ ]:
path_to_data = Path.cwd() / "data/input/fid_array_conc_2/"

In [ ]:
p = nmrpy.from_path(path_to_data)

#### EnzymeML document loading

In [ ]:
path_to_enzymeml = Path.cwd() / "data/output/cycloisomerization.json"

In [ ]:
p.parse_enzymeml_document(path_to_enzymeml)
CURRENT_MEASUREMENT = p.enzymeml_document.measurements[-1].id

#### Apodization, zero-filling, and Fourier transformation

It is common to improve the signal-to-noise ratio of NMR spectra by applying apodization. Also, the data has to be transformed into the frequency domain by Fourier transformation.

In [ ]:
p.emhz_fids(lb=0.3)     # Apodization
p.zf_fids()             # Zero-filling
p.ft_fids()             # Fourier transformation

In [ ]:
p.plot_array()

#### Phase correction, normalization, and realization

Oftentimes, the spectra need to be phase corrected, which can either be done automatically, using the `phase_correct_fids` method, or manually using the `phaser` widget. At this stage, the spectra can be realized, which discards the imaginary part of the spectra, and normalized by the maximum data value among all spectra.

In [ ]:
p.phase_correct_fids()  # Phase correction
p.real_fids()           # Discard imaginary part
p.norm_fids()           # Normalization

In [ ]:
p.plot_array()

#### Calibration

The spectra likely need to be calibrated by setting the ppm of a reference peak, which can be done using the `calibrate` method.

In this scenario, the spectra are already calibrated, so this step can be skipped.

In [ ]:
# p.calibrate()

#### Baseline correction

In case of a noisy baseline, it can be corrected using the `baseliner_fids` and `baseline_correct_fids` methods for setting and applying a baseline correction.

In [ ]:
p.baseliner_fids()

In [ ]:
p.baseline_correct_fids()

#### Peak picking

The peak assignment can be done using the `Fid.peakpicker` method or by manually setting the `Fid.peaks` and `Fid.ranges` attributes. The relevant peaks are:

- ≈6.71 ppm:      Mesitylene Singlet  (3 protons)
- ≈5.12-5.08 ppm: Educt Triplet       (1 proton)
- ≈3.14-3.13 ppm: Educt Doublet 1     (2 protons)
- ≈3.03-3.02 ppm: Educt Doublet 2     (2 protons)

In [ ]:
p.peakpicker()

#### Deconvolution

After picking the relevant peaks, these can be deconvoluted, making their integrals available for the subsequent analysis. There are also special plotting methods for visualizing the deconvolution results, such as `plot_deconv_array`.

In [ ]:
p.deconv_fids()         # Deconvolution

In [ ]:
p.plot_deconv_array(
    upper_ppm=5.25,
    lower_ppm=4.95,
    residual_colour=None,
    azim=-95,
    elev=20
)

#### Peak assignment

Species may be assigned to peaks either by using an EnzymeML document (the default) if available or by providing a list of species to the respective method through the `species_list` argument. The relevant species are:

- s5: Mesitylene
- s1: Educt
- s2: Product

When assigning species on the level of the entire FID array, an `index_list` of the FIDs to assign can be provided, effectively allowing to work on a slice of the FID array.

In [ ]:
p.assign_peaks()

---

### Analysis of the cycloisomerization reaction

#### Concentration calculation

Calculating the concentrations is highly dependent on the reaction in question and the spectra of the species involved. In this scenario, the integrals of the peaks corresponding to the educt and product are normalized by the integral of the mesitylene peak, which serves as an internal standard. The resulting values are then multiplied by the known concentration of mesitylene to obtain the concentrations of educt and product.

In [ ]:
def process_integrals(fid_array: "FidArray") -> dict[str, list[float]]:
    """Process the integrals of the peaks in the FID array."""
    FID_COUNT = len(fid_array.get_fids())

    int_dict = {
        "s5": np.zeros(FID_COUNT, dtype=float),
        "s1-t-1": np.zeros(FID_COUNT, dtype=float),
        "s1-t-2": np.zeros(FID_COUNT, dtype=float),
        "s1-t-3": np.zeros(FID_COUNT, dtype=float),
        "s1-d1-1": np.zeros(FID_COUNT, dtype=float),
        "s1-d1-2": np.zeros(FID_COUNT, dtype=float),
        "s1-d2-1": np.zeros(FID_COUNT, dtype=float),
        "s1-d2-2": np.zeros(FID_COUNT, dtype=float),
    }

    def peak_in_range(peak: float, target_range) -> bool:
        return target_range[0] >= peak >= target_range[1]

    for i, fid in enumerate(fid_array.get_fids()):
        ranges = fid.ranges
        peaks = fid.peaks
        integrals = fid.deconvoluted_integrals

        # Count how many peaks fall into each non-mesitylene range to detect triplet vs. doublets.
        range_stats = []
        for idx, r in enumerate(ranges[1:], start=1):
            count = sum(1 for p in peaks if peak_in_range(p, r))
            midpoint = (r[0] + r[1]) / 2
            range_stats.append({"idx": idx, "count": count, "mid": midpoint})

        triplet_range = next((r["idx"] for r in range_stats if r["count"] >= 3), None)
        doublet_ranges = [r for r in range_stats if r["idx"] != triplet_range and r["count"] > 0]
        doublet_ranges.sort(key=lambda r: r["mid"], reverse=True)
        d1_range = doublet_ranges[0]["idx"] if len(doublet_ranges) > 0 else None
        d2_range = doublet_ranges[1]["idx"] if len(doublet_ranges) > 1 else None

        t_filled = d1_filled = d2_filled = 0

        for integral, peak in zip(integrals, peaks):
            if peak_in_range(peak, ranges[0]):
                int_dict["s5"][i] = integral
                continue

            match_idx = None
            for idx in range(1, len(ranges)):
                if peak_in_range(peak, ranges[idx]):
                    match_idx = idx
                    break
            if match_idx is None:
                continue

            if triplet_range is not None and match_idx == triplet_range and t_filled < 3:
                t_filled += 1
                int_dict[f"s1-t-{t_filled}"][i] = integral
            elif d1_range is not None and match_idx == d1_range and d1_filled < 2:
                d1_filled += 1
                int_dict[f"s1-d1-{d1_filled}"][i] = integral
            elif d2_range is not None and match_idx == d2_range and d2_filled < 2:
                d2_filled += 1
                int_dict[f"s1-d2-{d2_filled}"][i] = integral

    return int_dict

def _sum_integrals(*integrals) -> float:
    """Sum the integrals of peaks."""
    return sum(integrals)

def _calculate_c_from_integrals(
    c_mesitylene: float, 
    mesitylene_integral: float, 
    educt_triplet_integral: float = None,
    educt_doublet1_integral: float = None,
    educt_doublet2_integral: float = None
) -> float:
    """Calculate the amount of educt from its integrals against the
    mesitylene integral.

    Args:
        c_mesitylene: The concentration of mesitylene in mol/l.
        mesitylene_integral: The integral of the mesitylene peak.
        educt_triplet_integral: The integral of the educt triplet peak.
        educt_doublet1_integral: The integral of the educt first doublet peak.
        educt_doublet2_integral: The integral of the educt second doublet peak.

    Returns:
        The amount of educt in mol/l.
    """
    if educt_triplet_integral is None and educt_doublet1_integral is None and educt_doublet2_integral is None:
        raise ValueError("At least one of the triplet, doublet1, or doublet2 integrals must be provided.")
    
    PROTONS_TRIPLET = 1
    PROTONS_DOUBLET1 = 2
    PROTONS_DOUBLET2 = 2
    PROTONS_MESITYLENE = 3

    sum_educt_integrals = 0
    sum_educt_protons = 0

    if educt_triplet_integral is not None:
        sum_educt_integrals += educt_triplet_integral
        sum_educt_protons += PROTONS_TRIPLET
    if educt_doublet1_integral is not None:
        sum_educt_integrals += educt_doublet1_integral
        sum_educt_protons += PROTONS_DOUBLET1
    if educt_doublet2_integral is not None:
        sum_educt_integrals += educt_doublet2_integral
        sum_educt_protons += PROTONS_DOUBLET2

    c_educt = ((sum_educt_integrals*PROTONS_MESITYLENE) / (mesitylene_integral*sum_educt_protons)) * c_mesitylene

    return c_educt

def calculate_educt_concentration(
    int_dict: dict[str, list[float]] = int_dict,
    enzml_doc: "EnzymeMLDocument" = p.enzymeml_document,
    measurement_id: str = CURRENT_MEASUREMENT,
) -> dict[str, float]:
    """
    """
    concentrations = {}
    concentrations["s5"] = []
    concentrations["s1"] = []

    for datum in enzml_doc.filter_measurements(id=measurement_id)[0].species_data:
        if datum.species_id == "s5":
            c_mesitylene = datum.prepared

    for i in range(len(int_dict["s5"])):
        concentrations["s5"].append(float(int_dict["s5"][i]))
        concentrations["s1"].append(
            float(
                _calculate_c_from_integrals(
                    c_mesitylene=c_mesitylene,
                    mesitylene_integral=int_dict["s5"][i],
                    educt_triplet_integral=_sum_integrals(
                        int_dict["s1-t-1"][i],
                        int_dict["s1-t-2"][i],
                        int_dict["s1-t-3"][i]
                    ) if "s1-t-1" in int_dict.keys() else None,
                    educt_doublet1_integral=_sum_integrals(
                        int_dict["s1-d1-1"][i],
                        int_dict["s1-d1-2"][i]
                    ) if "s1-d1-1" in int_dict.keys() else None,
                    educt_doublet2_integral=_sum_integrals(
                        int_dict["s1-d2-1"][i],
                        int_dict["s1-d2-2"][i]
                    ) if "s1-d2-1" in int_dict.keys() else None,
                )
            )
        )

    return concentrations

As the time array in the Bruker raw data is not correct, it has to be set manually using the `acqtime_array` parameter.

In [ ]:
t_2eq = [0.0, 2.5, 3.5, 5.5, 9.0, 19.0, 40.0]
p._params["acqtime_array"] = t_2eq

In [ ]:
int_dict = process_integrals(p)
concentrations = calculate_educt_concentration(int_dict)
p.concentrations = concentrations

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
for label, concentration in concentrations.items():
    if label == "s5":
        continue
    ax.plot(p.t, concentrations, 's', mec='k', label=label)
ax.legend()
ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (mM)')
ax.set_ylim(ymin=-0.0)

#### Apply the concentrations to EnzymeML

The concentrations calculated in NMRpy can be applied back to the original or any other EnzymeML document by calling the `FidArray.apply_to_enzymeml()` method. If no EnzymeML document is provided to the method, the original document is used by default.

In [ ]:
enzymeml_doc = p.apply_to_enzymeml(measurement_id=CURRENT_MEASUREMENT)

print(Pretty(enzymeml_doc.filter_measurements(id=CURRENT_MEASUREMENT)[0], max_length=5))

In [ ]:
pe.write_enzymeml(enzymeml_doc, path=path_to_enzymeml)

#### Save NMRPy data model

The NMRPy data model for the current measurement can be save as a JSON file, too. Thereby, all relevant NMR data and metadata are stored in a structured format.

In [ ]:
with open(f"./data/output/{p.enzymeml_document.filter_measurements(id=CURRENT_MEASUREMENT)[0].id}_data_model.json", "w") as f:
    f.write(p.data_model.model_dump_json(indent=2))

#### Save the NMRPy library state

The entire state of the NMRPy library, including all loaded data and calculated results, can be saved in a pickle file, allowing to easily reload the state at a later point in time without having to repeat the data loading and processing steps.

In [ ]:
p.save_to_file(f"./data/state/{p.enzymeml_document.filter_measurements(id=CURRENT_MEASUREMENT)[0].id}.nmrpy")

---

### Disclosure

**Contributions**

If you wish to contribute to the EnyzmeML and/or NMRPy platforms, find us on our [EnzymeML GitHub](https://github.com/EnzymeML) and [NMRPy GitHub](https://github.com/NMRPy)!

**BSD 3-Clause License**

Copyright (c) 2026 Torsten Giess

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS “AS IS” AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.